In [1]:
import sys
import pickle
import torch
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.append("..")
from src.data.preprocessing import build_leave_one_out
from src.models.gru4rec import GRU4Rec
from src.losses.sampled_ranking_loss import BPRLoss
from src.trainers.gru4rec_trainer import GRU4RecTrainer
from src.data.dataset import SessionDataset, collate_sessions

In [2]:
PROCESSED_DIR = Path("../data/processed")

with open(PROCESSED_DIR / "sequences.pkl", "rb") as f:
    sequences = pickle.load(f)

with open(PROCESSED_DIR / "item_mappings.pkl", "rb") as f:
    mappings = pickle.load(f)
vocab_size = mappings["vocab_size"]

print(f"Users: {len(sequences)}, Vocab size: {vocab_size}")

Users: 197519, Vocab size: 47976


In [3]:
split_data = build_leave_one_out(sequences)
print(f"After split: {len(split_data)} users")

train_prefixes = {uid: data["train"] for uid, data in split_data.items()}

After split: 197519 users


In [4]:
dataset = SessionDataset(train_prefixes)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_sessions)
print(f"Number of batches: {len(dataloader)}")

Number of batches: 6173


In [5]:
DEVICE = "cpu"
DEVICE

'cpu'

In [6]:
embed_dim = 64
hidden_dim = 128
num_negs = 10
lr = 1e-3
weight_decay = 0.001
epochs = 10

model = GRU4Rec(vocab_size, embed_dim, hidden_dim).to(DEVICE)
bpr_loss = BPRLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

trainer = GRU4RecTrainer(model, bpr_loss, optimizer, vocab_size, DEVICE, num_negs=num_negs)

In [7]:
best_recall = -1
best_model_state = None

for epoch in range(1, epochs + 1):
    loss = trainer.train_epoch(dataloader)
    val_metrics = trainer.evaluate(split_data, target_key="val_target", k=5)
    recall = val_metrics["Recall@5"]
    print(f"Epoch {epoch}: loss={loss:.4f}, val Recall@5={recall:.4f}, val NDCG@5={val_metrics['NDCG@5']:.4f}, val MRR={val_metrics['MRR']:.4f}")

    if recall > best_recall:
        best_recall = recall
        best_model_state = model.state_dict().copy()
        torch.save(best_model_state, PROCESSED_DIR / "gru4rec_best.pt")
        print(f"New best model saved (Recall@5={best_recall:.4f})")

Training: 100%|██████████| 6173/6173 [13:42<00:00,  7.50it/s]


Epoch 1: loss=0.1649, val Recall@5=0.0556, val NDCG@5=0.0382, val MRR=0.0433
New best model saved (Recall@5=0.0556)


Training: 100%|██████████| 6173/6173 [13:45<00:00,  7.48it/s]


Epoch 2: loss=0.0978, val Recall@5=0.0621, val NDCG@5=0.0423, val MRR=0.0471
New best model saved (Recall@5=0.0621)


Training: 100%|██████████| 6173/6173 [13:50<00:00,  7.43it/s]


...

Epoch 8: loss=0.0582, val Recall@5=0.0806, val NDCG@5=0.0546, val MRR=0.0601
New best model saved (Recall@5=0.0806)


Training: 100%|██████████| 6173/6173 [14:01<00:00,  7.33it/s]


Epoch 9: loss=0.0556, val Recall@5=0.0822, val NDCG@5=0.0556, val MRR=0.0610
New best model saved (Recall@5=0.0822)


Training: 100%|██████████| 6173/6173 [13:44<00:00,  7.49it/s]


Epoch 10: loss=0.0535, val Recall@5=0.0835, val NDCG@5=0.0567, val MRR=0.0622
New best model saved (Recall@5=0.0835)


In [8]:
model.load_state_dict(torch.load(PROCESSED_DIR / "gru4rec_best.pt", map_location=DEVICE))
test_metrics = trainer.evaluate(split_data, target_key="test_target", k=5)
print("Test results for best model:")
print(test_metrics)

Test results for best model:
{'Recall@5': np.float64(0.07167411742667794), 'NDCG@5': np.float64(0.04901791748260631), 'MRR': np.float64(0.05464705945832034)}
